In [1]:
##################
### Keagan Moo ###
### 2022/07/XX ###
### Linear Mod ###
##################
# Based on the prior script applied to ATAC seq peaks, metadata has been changed to Arushi update for RNA nuclei

# We don't have TSS Enrichment from the RNA nuclei, so they've been removed from the model... obviously.

options(stringsAsFactors = F)
suppressPackageStartupMessages(library(edgeR))
suppressPackageStartupMessages(library(glue))
suppressPackageStartupMessages(library(data.table))
suppressPackageStartupMessages(library(glue))
#library(zoo)
suppressPackageStartupMessages(library(forecast))
#library(reshape2)
#library(parallel)
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(statmod))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(DESeq2))
#suppressPackageStartupMessages(library(MAST))

In [2]:
#features <- c("Type_1", "Type_2a", "Type_2x", "Endothelial",
#             "Mesenchymal_Stem_Cell", "Smooth_Muscle")
feature <- "Type_1"

In [4]:
metadata.factors <- read.table(file = "/nfs/turbo/umms-scjp/kgmoo/data/fusion_sample_info.tsv", header = TRUE, sep = "\t")
head(metadata.factors)
nucMet <- read.table("/nfs/turbo/umms-scjp/kgmoo/data/cluster_info_qc.tsv", sep = "\t", header = TRUE)
nucMet <- nucMet[(nucMet$modality == "rna") & (nucMet$demux_cat != "multiome"),]
nucMet <- nucMet %>%
  group_by(sample) %>%
  mutate(tss_enrich_median = median(tss_enrichment, na.rm = T),
         n_nuclei = n())
metadata.factors <- merge(metadata.factors, nucMet, by = "SNG.1ST") %>% select(c(SNG.1ST, sample, batch.x, cohort, modality, age, sex, bmi, tss_enrich_median, n_nuclei, coarse_cluster_name)) %>% distinct()
morMet <- read.table("/nfs/turbo/umms-scjp/kgmoo/data/batch-sample-info-full.tsv", sep = "\t", header = TRUE)
length(unique(metadata.factors$sample))
metadata.factors <- merge(metadata.factors, morMet, by = "SNG.1ST") %>% select(c(SNG.1ST, sample.x, batch.x, cohort.x, modality, age.x, sex.x, bmi.x, area_name, clamp, fins, tss_enrich_median, n_nuclei, ogtt, coarse_cluster_name))
length(unique(metadata.factors$sample.x))
saveRDS(metadata.factors, file = "/nfs/turbo/umms-scjp/kgmoo/data/skeletalmuscleRNA_metadata.rds")
# Do we need to add the cell type specific tss_enirchment data and does it need a different set than the file for atac that arushi provided???

metadata.factorsComp <- metadata.factors[metadata.factors$coarse_cluster_name %in% c("Type_1", "Type_2a", "Type_2x"),]$sample.x
metadata.factorsOut <- metadata.factors[metadata.factors$age.x < 40,]$sample.x
metadata.factors <- metadata.factors[metadata.factors$coarse_cluster_name == glue("{feature}") & (metadata.factors$sample.x %in% metadata.factorsComp),]
metadata.factorsOld <- metadata.factors[metadata.factors$coarse_cluster_name == glue("{feature}") & (metadata.factors$sample.x %in% metadata.factorsComp) & !(metadata.factors$sample.x %in% metadata.factorsOut),]

,SNG.1ST,age,sex,batch,bmi
,<chr>,<dbl>,<chr>,<chr>,<dbl>
1,22008,63,M,499-NM,30.03386
2,12094,57,M,499-NM,29.54981
3,32001,61,M,499-NM,24.21258
4,32063,72,F,499-NM,28.51239
5,12095,70,M,499-NM,20.33166
6,12050,69,F,499-NM,31.87500


[1] 287

[1] 287

In [18]:
length(metadata.factorsComp)
dim(metadata.factors)
metadata.factors

[1] 858

[1] 287  15

,SNG.1ST,sample.x,batch.x,cohort.x,modality,age.x,sex.x,bmi.x,area_name,clamp,fins,tss_enrich_median,n_nuclei,ogtt,coarse_cluster_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<int>,<chr>,<chr>
2,12001,12001--510,510-NM,FUSION,rna,66,M,30.37742,Kuopio,NA,8.2,NA,744,IGT,Type_1
15,12002,12002--551,551-NM,FUSION,rna,61,M,25.10174,Kuopio,NA,3.5,NA,299,NGT,Type_1
24,12003,12003--530,530-NM,FUSION,rna,58,M,25.51226,Kuopio,NA,4.2,NA,1148,NGT,Type_1
36,12004,12004--530,530-NM,FUSION,rna,64,M,23.03005,Kuopio,NA,3.7,NA,842,NGT,Type_1
47,12006,12006--530,530-NM,FUSION,rna,56,F,22.79218,Kuopio,NA,6.4,NA,752,NGT,Type_1
60,12007,12007--647,647-NM,FUSION,rna,64,F,21.35780,Kuopio,NA,1.8,NA,598,IGT,Type_1
71,12008,12008--530,530-NM,FUSION,rna,65,M,23.80946,Kuopio,NA,7.1,NA,887,NGT,Type_1
83,12009,12009--499,499-NM,FUSION,rna,68,M,24.36640,Kuopio,NA,4.1,NA,732,NGT,Type_1
95,12010,12010--647,647-NM,FUSION,rna,58,F,32.00195,Kuopio,NA,7.9,NA,12,IGT,Type_1


In [5]:
metadata.factors[metadata.factors$age < 40,]
head(metadata.factors[metadata.factors$coarse_cluster_name == feature,])
head(nucMet)
#metadata.factors[metadata.factors$sample %in% c("SA02842--564","SA02842--625"),] checking for sample Ricardo called "sex missassigned"
#mutate(sex = case_when(sample %in% c("SA02842--564", "SA02842--625") ~ "M",
#                         T ~ sex))

,SNG.1ST,sample.x,batch.x,cohort.x,modality,age.x,sex.x,bmi.x,area_name,clamp,fins,tss_enrich_median,n_nuclei,ogtt,coarse_cluster_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<int>,<chr>,<chr>
1351,12166,12166--647,647-NM,FUSION,rna,33,F,28.92026,Kuopio,NA,12.7,NA,592,NGT,Type_1
1384,12173,12173--564,564-NM,FUSION,rna,25,M,24.67691,Kuopio,NA,2.9,NA,397,NGT,Type_1
2653,32071,32071--692,692-NM,FUSION,rna,20,F,23.32342,Helsinki,NA,11.5,NA,499,NGT,Type_1


,SNG.1ST,sample.x,batch.x,cohort.x,modality,age.x,sex.x,bmi.x,area_name,clamp,fins,tss_enrich_median,n_nuclei,ogtt,coarse_cluster_name
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<int>,<chr>,<chr>
2,12001,12001--510,510-NM,FUSION,rna,66,M,30.37742,Kuopio,NA,8.2,NA,744,IGT,Type_1
15,12002,12002--551,551-NM,FUSION,rna,61,M,25.10174,Kuopio,NA,3.5,NA,299,NGT,Type_1
24,12003,12003--530,530-NM,FUSION,rna,58,M,25.51226,Kuopio,NA,4.2,NA,1148,NGT,Type_1
36,12004,12004--530,530-NM,FUSION,rna,64,M,23.03005,Kuopio,NA,3.7,NA,842,NGT,Type_1
47,12006,12006--530,530-NM,FUSION,rna,56,F,22.79218,Kuopio,NA,6.4,NA,752,NGT,Type_1
60,12007,12007--647,647-NM,FUSION,rna,64,F,21.35780,Kuopio,NA,1.8,NA,598,IGT,Type_1


index,UMAP_1,UMAP_2,modality,library,barcode,batch,cluster,demux_cat,SNG.1ST,hqaa_umi,fraction_mitochondrial,cohort,sample,tss_enrichment,log10_hqaa_umi,coarse_cluster_name,tss_enrich_median,n_nuclei
<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<dbl>,<int>
rna.499.NM.1.AAACCCAAGCCGATCC,1.627538,13.069763,rna,499-NM-1,AAACCCAAGCCGATCC,499,0,SNG-SNG,12055,10114,4.079848e-05,FUSION,12055--499,NA,9.221676,Type_1,NA,1186
rna.499.NM.1.AAACCCATCCCAGGAC,-7.560267,-7.390422,rna,499-NM-1,AAACCCATCCCAGGAC,499,1,SNG-SNG,32063,8161,4.793547e-04,FUSION,32063--499,NA,9.007122,Type_2a,NA,1076
rna.499.NM.1.AAACGCTTCCGGACTG,-7.056693,3.126394,rna,499-NM-1,AAACGCTTCCGGACTG,499,2,SNG-SNG,22008,16635,3.470795e-04,FUSION,22008--499,NA,9.719264,Type_2x,NA,580
rna.499.NM.1.AAAGAACAGCCTCTTC,2.234093,-14.239776,rna,499-NM-1,AAAGAACAGCCTCTTC,499,3,SNG-SNG,32084,5398,1.254688e-03,FUSION,32084--499,NA,8.593784,Endothelial,NA,1265
rna.499.NM.1.AAAGAACCAACAAGAT,-1.786401,-6.624691,rna,499-NM-1,AAAGAACCAACAAGAT,499,1,SNG-SNG,12095,6565,8.294554e-04,FUSION,12095--499,NA,8.789508,Type_2a,NA,609
rna.499.NM.1.AAAGAACGTTCTAAGC,5.073321,2.630244,rna,499-NM-1,AAAGAACGTTCTAAGC,499,0,SNG-SNG,12086,7336,1.262723e-03,FUSION,12086--499,NA,8.900549,Type_1,NA,780


In [5]:
peaksDF <- read.table(glue("/nfs/turbo/umms-scjp/kgmoo/data/rna_sample_counts/{feature}_rna_postDecontx_postArushiQC.csv"), sep = ",", header = TRUE)
Ens2Sym <- peaksDF %>% dplyr::select(GeneID, Symbol)
peaks <- peaksDF %>% tibble::column_to_rownames("X") %>% dplyr::select(-GeneID, -Symbol)
peaks <- peaks[,colnames(peaks)]
colnames(peaks) <- gsub("X","", colnames(peaks))
metadata.factors$sample <- gsub("-", ".", metadata.factors$sample)
metadata.factorsOld$sample <- gsub("-", ".", metadata.factorsOld$sample)
peaks <- peaks[,colnames(peaks) %in% metadata.factors$sample]
peaksOld <- peaks[,colnames(peaks) %in% metadata.factorsOld$sample]
metadata.factors$n_nuclei_l10 <- log(metadata.factors$n_nuclei, 10)
metadata.factorsOld$n_nuclei_l10 <- log(metadata.factorsOld$n_nuclei, 10)
metadata.factors <- metadata.factors %>% arrange(match(sample, factor(colnames(peaks))))
metadata.factorsOld <- metadata.factorsOld %>% arrange(match(sample, factor(colnames(peaksOld))))
colnames(metadata.factors) <- c("SNG.1ST", "sample.x", "batch", "cohort", "modality", "age", "sex", "bmi", "area_name", "clamp", "fins", "tss_enrich_median", "n_nuclei", "ogtt", "coarse_cluster_name", "sample", "n_nuclei_l10")
colnames(metadata.factorsOld) <- c("SNG.1ST", "sample.x", "batch", "cohort", "modality", "age", "sex", "bmi", "area_name", "clamp", "fins", "tss_enrich_median", "n_nuclei", "ogtt", "coarse_cluster_name", "sample", "n_nuclei_l10")

Warning message in file(file, "rt"):
“cannot open file '/nfs/turbo/umms-scjp/kgmoo/data/rna_sample_counts/Type_1_rna_postDecontx_postArushiQC.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


In [6]:
head(peaksDF)
dim(peaks)

,X,GeneID,Symbol,X12055..499,X32063..499,X22008..499,X32084..499,X12095..499,X12086..499,X12074..499,⋯,X12174..692,X22013..692,X22031..692,X32109..692,X12078..692,X32113..692,X32069..692,X22009..692,X22041..692,X32119..692
,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,ENSG00000223972.5,ENSG00000223972.5,DDX11L1,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
2,ENSG00000227232.5,ENSG00000227232.5,WASH7P,5,6,1,3,3,1,1,⋯,3,4,8,4,6,0,2,1,0,0
3,ENSG00000278267.1,ENSG00000278267.1,MIR6859-1,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
4,ENSG00000243485.5,ENSG00000243485.5,MIR1302-2HG,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
5,ENSG00000284332.1,ENSG00000284332.1,MIR1302-2,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
6,ENSG00000237613.2,ENSG00000237613.2,FAM138A,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


[1] 58962   287

In [5]:
nms <- c("p0", "p0.1", "p0.2", "p0.3", "p0.4", "p0.5")
pcts <- c(0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5)
cnts <- c(1, 2, 5, 10, 20, 50, 100)
ckFrame <- data.frame(percent <- 0, counts <- 0, genes <- 58962)

for (pct in pcts) {
    #print(pct)
    temp <- c()
    checkSize <- round(nrow(metadata.factors)*pct)
    for (cnt in cnts) {
        #print(cnt)
        keep <- rowSums(peaks >= cnt) >= checkSize
        tempVal <- dim(peaks[keep,])[1]
        #print(tempVal)
        ckFrame <- rbind(ckFrame, c(pct, cnt, tempVal))
    }
        
}
colnames(ckFrame) <- c("percents","counts","genes")
ckFrame$counts

[1]   0   1   2   5  10  20  50 100   1   2   5  10  20  50 100   1   2   5  10
[20]  20  50 100   1   2   5  10  20  50 100   1   2   5  10  20  50 100   1   2
[39]   5  10  20  50 100   1   2   5  10  20  50 100   1   2   5  10  20  50 100

In [9]:
head(ckFrame)

,percents,counts,genes
,<dbl>,<dbl>,<dbl>
1,0.00,0,58962
2,0.05,1,30388
3,0.05,2,22351
4,0.05,5,15438
5,0.05,10,11985
6,0.05,20,9103


In [6]:
pdf(glue("/scratch/scjp_root/scjp1/kgmoo/analysis/20240814_kgmoo_RNAge/plots/GeneFilterCheck_Type2a.pdf"), width=6, height=6)
mygg <- ggplot(data=ckFrame, aes(x=percents, y=genes, color = counts, fill = counts, justification = -.1)) +
  #ggdist::stat_halfeye(alpha = 0.75, position = position_nudge(x=0.1), adjust = 1, width = 0.8, .width = 0)+
  #scale_fill_manual(values = c("#FF0000","#0000FF","#000000"))+
  geom_point() +
  theme_bw() + #scale_color_manual(values = c("#FF0000","#0000FF","#000000")) +
  xlab("Min. proportions of samples meeting count threshold") + ylab("Number of genes") + ggtitle("Gene filter screening")
mygg
dev.off()

png 
  2

In [6]:
checkSize <- round(nrow(metadata.factors)*0.1)
keep <- rowSums(peaks >= 10) >= checkSize
peaks <- peaks[keep,]

#checkSize <- round(nrow(metadata.factorsOld)*0.1)
#keep <- rowSums(peaksOld >= 10) >= checkSize
#peaksOld <- peaksOld[keep,]

In [7]:
dim(metadata.factors)
dim(peaks)

[1] 287  17

[1] 13885   287

In [8]:
libsizes <- colSums(peaks)
if (all(names(libsizes) == metadata.factors$sample)) {
  metadata.factors$libsize <- log2(unname(libsizes))
} else {
  stop("Something wrong (2)")
}
peaks <- peaks %>% data.matrix()

#libsizesOld <- colSums(peaksOld)
#if (all(names(libsizesOld) == metadata.factorsOld$sample)) {
#  metadata.factorsOld$libsize <- log2(unname(libsizesOld))
#} else {
#  stop("Something wrong (2)")
#}
#peaksOld <- peaksOld %>% data.matrix()

In [9]:
### PRE-NORMALIZE COUNTS ###

sampleDepths <- colSums(peaks)
metadata.factors$sDepth <- sampleDepths

# TMM Normalize Data (If we want to use adjusted counts downstream)
fPeaks <- peaks[,colnames(peaks) %in% metadata.factors$sample[metadata.factors$sex == "F"]]
mPeaks <- peaks[,colnames(peaks) %in% metadata.factors$sample[metadata.factors$sex == "M"]]

libsizes <- colSums(peaks)
fLibsizes <- colSums(fPeaks)
mLibsizes <- colSums(mPeaks)

peaksDGE <- DGEList(peaks,lib.size = libsizes)
fPeaksDGE <- DGEList(fPeaks,lib.size = fLibsizes)
mPeaksDGE <- DGEList(mPeaks,lib.size = mLibsizes)

peaksDGE <- calcNormFactors(peaksDGE,method = "TMM")
fPeaksDGE <- calcNormFactors(fPeaksDGE,method = "TMM")
mPeaksDGE <- calcNormFactors(mPeaksDGE,method = "TMM")

peaksNorm <- peaks %*% diag(peaksDGE$samples$norm.factors)
fPeaksNorm <- fPeaks %*% diag(fPeaksDGE$samples$norm.factors)
mPeaksNorm <- mPeaks %*% diag(mPeaksDGE$samples$norm.factors)

colnames(peaksNorm) <- colnames(peaks)
colnames(fPeaksNorm) <- colnames(fPeaks)
colnames(mPeaksNorm) <- colnames(mPeaks)

peaksNormL2 <- log2(1+peaksNorm)
fPeaksNormL2 <- log2(1+fPeaksNorm)
mPeaksNormL2 <- log2(1+mPeaksNorm)

peaksNormMean <- matrix(rep(rowMeans(peaksNorm),ncol(peaksNorm)),nrow = nrow(peaksNorm))
fPeaksNormMean <- matrix(rep(rowMeans(fPeaksNorm),ncol(fPeaksNorm)),nrow = nrow(fPeaksNorm))
mPeaksNormMean <- matrix(rep(rowMeans(mPeaksNorm),ncol(mPeaksNorm)),nrow = nrow(mPeaksNorm))

In [13]:
### PRE-NORMALIZE COUNTS (OLD)###

#sampleDepthsOld <- colSums(peaksOld)
#metadata.factorsOld$sDepth <- sampleDepthsOld

# TMM Normalize Data (If we want to use adjusted counts downstream)
#fPeaksOld <- peaksOld[,colnames(peaksOld) %in% metadata.factorsOld$sample[metadata.factorsOld$sex == "F"]]
#mPeaksOld <- peaksOld[,colnames(peaksOld) %in% metadata.factorsOld$sample[metadata.factorsOld$sex == "M"]]

#libsizesOld <- colSums(peaksOld)
#fLibsizesOld <- colSums(fPeaksOld)
#mLibsizesOld <- colSums(mPeaksOld)

#peaksDGEOld <- DGEList(peaksOld,lib.size = libsizesOld)
#fPeaksDGEOld <- DGEList(fPeaksOld,lib.size = fLibsizesOld)
#mPeaksDGEOld <- DGEList(mPeaksOld,lib.size = mLibsizesOld)

#peaksDGEOld <- calcNormFactors(peaksDGEOld,method = "TMM")
#fPeaksDGEOld <- calcNormFactors(fPeaksDGEOld,method = "TMM")
#mPeaksDGEOld <- calcNormFactors(mPeaksDGEOld,method = "TMM")

#peaksNormOld <- peaksOld %*% diag(peaksDGEOld$samples$norm.factors)
#fPeaksNormOld <- fPeaksOld %*% diag(fPeaksDGEOld$samples$norm.factors)
#mPeaksNormOld <- mPeaksOld %*% diag(mPeaksDGEOld$samples$norm.factors)

#colnames(peaksNormOld) <- colnames(peaksOld)
#colnames(fPeaksNormOld) <- colnames(fPeaksOld)
#colnames(mPeaksNormOld) <- colnames(mPeaksOld)

#peaksNormL2Old <- log2(1+peaksNormOld)
#fPeaksNormL2Old <- log2(1+fPeaksNormOld)
#mPeaksNormL2Old <- log2(1+mPeaksNormOld)

#peaksNormMeanOld <- matrix(rep(rowMeans(peaksNormOld),ncol(peaksNormOld)),nrow = nrow(peaksNormOld))
#fPeaksNormMeanOld <- matrix(rep(rowMeans(fPeaksNormOld),ncol(fPeaksNormOld)),nrow = nrow(fPeaksNormOld))
#mPeaksNormMeanOld <- matrix(rep(rowMeans(mPeaksNormOld),ncol(mPeaksNormOld)),nrow = nrow(mPeaksNormOld))

In [10]:
fMeta <- metadata.factors[metadata.factors$sex == "F",]
mMeta <- metadata.factors[metadata.factors$sex == "M",]

fMetaOld <- metadata.factorsOld[metadata.factorsOld$sex == "F",]
mMetaOld <- metadata.factorsOld[metadata.factorsOld$sex == "M",]

In [11]:
### LINEAR MODEL WITH COVARIATES (DEPREACATED) ###
designNull <- "model.matrix(~bmi + n_nuclei_l10 + batch + ogtt + offset(libsize))"
designAge <- "model.matrix(~bmi + n_nuclei_l10 + batch + ogtt + offset(libsize) + age)"
designNullParsed <- with(metadata.factors, eval(parse(text=designNull)))
fDesignNullParsed <- with(fMeta, eval(parse(text=designNull)))
mDesignNullParsed <- with(mMeta, eval(parse(text=designNull)))
designAgeParsed <- with(metadata.factors, eval(parse(text=designAge)))
fDesignAgeParsed <- with(fMeta, eval(parse(text=designAge)))
mDesignAgeParsed <- with(mMeta, eval(parse(text=designAge)))

peaksDGE <- estimateDisp(peaksDGE, designNullParsed, robust=T,tagwise = T,trend.method = "locfit")
fPeaksDGE <- estimateDisp(fPeaksDGE, fDesignNullParsed, robust=T,tagwise = T,trend.method = "locfit")
mPeaksDGE <- estimateDisp(mPeaksDGE, mDesignNullParsed, robust=T,tagwise = T,trend.method = "locfit")
#peaksDAGE <- estimateDisp(peaksDGE, designAgeParsed, robust=T,tagwise = T,trend.method = "locfit")
fPeaksDAGE <- estimateDisp(fPeaksDGE, fDesignAgeParsed, robust=T,tagwise = T,trend.method = "locfit")
mPeaksDAGE <- estimateDisp(mPeaksDGE, mDesignAgeParsed, robust=T,tagwise = T,trend.method = "locfit")

designNullGLM <- glmFit(peaksDGE, designNullParsed)
fDesignNullGLM <- glmFit(fPeaksDGE, fDesignNullParsed)
mDesignNullGLM <- glmFit(mPeaksDGE, mDesignNullParsed)
fDesignAgeGLM <- glmFit(fPeaksDAGE, fDesignAgeParsed)
mDesignAgeGLM <- glmFit(mPeaksDAGE, mDesignAgeParsed)

#fDesignAgeLM <- lm(fPeaksDGE$counts ~ metadata.factors$bmi + metadata.factors$tss_enrich_median + metadata.factors$n_nuclei_l10 + metadata.factors$batch)
#mDesignAgeLM <- lm(fPeaksDGE, mDesignAgeParsed)

# Need to specify which coeffecient to test, apparently it defualts to "THE LAST"?
# Ideally we would look into specifying contrast though we've got the age coeff for now
fRes <- topTags(glmLRT(fDesignAgeGLM), n = 200000000)$table
mRes <- topTags(glmLRT(mDesignAgeGLM), n = 200000000)$table

peaksAdjusted <- peaksNormL2 - log2(1+designNullGLM$fitted.values) + peaksNormMean
fPeaksAdjusted <- fPeaksNormL2 - log2(1+fDesignNullGLM$fitted.values) + fPeaksNormMean
mPeaksAdjusted <- mPeaksNormL2 - log2(1+mDesignNullGLM$fitted.values) + mPeaksNormMean

In [16]:
head(fMeta)
dim(fPeaks)

,SNG.1ST,sample.x,batch,cohort,modality,age,sex,bmi,area_name,clamp,fins,tss_enrich_median,n_nuclei,ogtt,coarse_cluster_name,sample,n_nuclei_l10,libsize,sDepth
,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
1,12055,12055--499,499-NM,FUSION,rna,53,F,22.77518,Kuopio,NA,4.2,NA,1186,NGT,Type_1,12055..499,3.074085,22.45965,5768035
2,32063,32063--499,499-NM,FUSION,rna,72,F,28.51239,Helsinki,NA,7.9,NA,1076,IGT,Type_1,32063..499,3.031812,22.66024,6628470
4,32084,32084--499,499-NM,FUSION,rna,70,F,23.44286,Helsinki,NA,4.7,NA,1265,IFG,Type_1,32084..499,3.102091,22.80280,7316878
9,22019,22019--499,499-NM,FUSION,rna,56,F,26.47490,Savitaipale,NA,6.9,NA,790,NGT,Type_1,22019..499,2.897627,21.43578,2836691
15,12040,12040--499,499-NM,FUSION,rna,60,F,28.85444,Kuopio,NA,9.3,NA,630,NGT,Type_1,12040..499,2.799341,21.42047,2806759
16,12180,12180--499,499-NM,FUSION,rna,68,F,23.96288,Kuopio,NA,8.1,NA,589,NGT,Type_1,12180..499,2.770115,21.22006,2442720


[1] 13960   118

In [12]:
fdds <- DESeqDataSetFromMatrix(countData = fPeaks,
                              colData = fMeta,
                              design= ~bmi + n_nuclei_l10 + batch + ogtt + age)
fdds <- DESeq(fdds)
fDESeqRes <- results(fdds, name = "age")

mdds <- DESeqDataSetFromMatrix(countData = mPeaks,
                              colData = mMeta,
                              design= ~bmi + n_nuclei_l10 + batch + ogtt + age)
mdds <- DESeq(mdds)
mDESeqRes <- results(mdds, name = "age")

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names i

In [13]:
#fddsOld <- DESeqDataSetFromMatrix(countData = fPeaksOld,
#                              colData = fMetaOld,
#                              design= ~bmi + n_nuclei_l10 + batch + ogtt + age)
#fddsOld <- DESeq(fddsOld)
#fDESeqResOld <- results(fddsOld, name = "age")

#mddsOld <- DESeqDataSetFromMatrix(countData = mPeaksOld,
#                              colData = mMetaOld,
#                              design= ~bmi + n_nuclei_l10 + batch + ogtt + age)
#mddsOld <- DESeq(mddsOld)
#mDESeqResOld <- results(mddsOld, name = "age")

Warning message in DESeqDataSet(se, design = design, ignoreRank):
“some variables in design formula are characters, converting to factors”
  the design formula contains one or more numeric variables with integer values,
  specifying a model with increasing fold change for higher values.
  did you mean for this to be a factor? if so, first convert
  this variable to a factor using the factor() function

  the design formula contains one or more numeric variables that have mean or
  standard deviation larger than 5 (an arbitrary threshold to trigger this message).
  Including numeric variables with large mean can induce collinearity with the intercept.
  Users should center and scale numeric variables in the design to improve GLM convergence.

  Note: levels of factors in the design contain characters other than
  letters, numbers, '_' and '.'. It is recommended (but not required) to use
  only letters, numbers, and delimiters '_' or '.', as these are safe characters
  for column names i

`fData` has no primerid.  I'll make something up.

`cData` has no wellKey.  I'll make something up.

Warning message in .sparse2dense(e1):
“sparse->dense coercion: allocating vector of size 79.3 GiB”


,logFC,logCPM,LR,PValue,FDR
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ENSG00000049130.15,-0.03078516,7.877498,17.93135,2.290176e-05,0.1261887
ENSG00000166510.14,0.03013872,7.276808,16.36382,5.227370e-05,0.1440141
ENSG00000115310.18,-0.02028415,8.212723,14.63867,1.302151e-04,0.2369384
ENSG00000104660.18,0.03898019,6.188087,14.11457,1.720061e-04,0.2369384
ENSG00000089053.13,0.02179191,7.400580,13.28533,2.674912e-04,0.2835793
ENSG00000147650.11,0.03904895,6.044839,12.61912,3.818219e-04,0.2835793


log2 fold change (MLE): age 
Wald test p-value: age 
DataFrame with 6 rows and 6 columns
                    baseMean log2FoldChange      lfcSE       stat    pvalue
                   <numeric>      <numeric>  <numeric>  <numeric> <numeric>
ENSG00000230021.10   5.01477   -0.004504500 0.00978881 -0.4601683  0.645395
ENSG00000237491.9    3.84058   -0.008688067 0.01255882 -0.6917902  0.489069
ENSG00000131591.17   3.46886    0.008878362 0.01289592  0.6884627  0.491161
ENSG00000221978.12  13.50091    0.005490136 0.00526306  1.0431448  0.296881
ENSG00000248333.8    3.75142    0.008665033 0.01108940  0.7813801  0.434579
ENSG00000078369.18  47.17316   -0.000311064 0.00328919 -0.0945715  0.924655
                        padj
                   <numeric>
ENSG00000230021.10  0.983060
ENSG00000237491.9   0.970729
ENSG00000131591.17  0.970729
ENSG00000221978.12  0.948157
ENSG00000248333.8   0.963261
ENSG00000078369.18  0.996814

In [13]:
saveRDS(list(fRes, mRes) , glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240814_kgmoo_RNAge/linear_per_cluster/{feature}_glmoff.rds"))
saveRDS(list(peaksAdjusted, fPeaksAdjusted, mPeaksAdjusted), glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240814_kgmoo_RNAge/linear_per_cluster/{feature}_glmoffadj.rds"))
peaky <- readRDS(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240814_kgmoo_RNAge/linear_per_cluster/{feature}_glmoffadj.rds"))
peaksAdjusted <- peaky[[1]]

nrow(fRes[fRes$PValue <= 0.05,])
nrow(mRes[mRes$PValue <= 0.05,])
nrow(fRes[fRes$FDR <= 0.05,])
nrow(mRes[mRes$FDR <= 0.05,])

[1] 1377

[1] 966

[1] 17

[1] 0

In [14]:
saveRDS(list(fDESeqRes, mDESeqRes) , glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20240814_kgmoo_RNAge/linear_per_cluster/{feature}_DESeq.rds"))

fDESeqRes$padj[is.na(fDESeqRes$padj)] <- 1
mDESeqRes$padj[is.na(mDESeqRes$padj)] <- 1

nrow(fDESeqRes[fDESeqRes$pvalue <= 0.05,])
nrow(mDESeqRes[mDESeqRes$pvalue <= 0.05,])
nrow(fDESeqRes[fDESeqRes$padj <= 0.1,])
nrow(mDESeqRes[mDESeqRes$padj <= 0.1,])

[1] 1313

[1] 972

[1] 76

[1] 13

In [20]:
fDESeqResDF <- data.frame(fDESeqRes)
mDESeqResDF <- data.frame(mDESeqRes)
nrow(fDESeqResDF[(fDESeqResDF$log2FoldChange < 0) & (fDESeqResDF$pvalue < 0.05),])
nrow(fDESeqResDF[(fDESeqResDF$log2FoldChange > 0) & (fDESeqResDF$pvalue < 0.05),])
nrow(mDESeqResDF[(mDESeqResDF$log2FoldChange < 0) & (fDESeqResDF$pvalue < 0.05),])
nrow(mDESeqResDF[(mDESeqResDF$log2FoldChange > 0) & (fDESeqResDF$pvalue < 0.05),])

[1] 418

[1] 542

[1] 453

[1] 507

In [10]:
Test <- readRDS(glue("/scratch/scjp_root/scjp1/kgmoo/analysis/20240814_kgmoo_RNAge/linear_per_cluster/{feature}_DESeq.rds"))
Test <- Test[[1]]
saveRDS(data.frame(Test), file = "/scratch/scjp_root/scjp1/kgmoo/analysis/20240814_kgmoo_RNAge/linear_per_cluster/Test_DESeq.rds")

In [17]:
CompDF <- merge(fRes,data.frame(fDESeqRes),by = "row.names")
head(CompDF)

,Row.names,logFC,logCPM,LR,PValue,FDR,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
,<I<chr>>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,ENSG00000000419.12,0.0064772510,5.763621,4.622640460,0.03155263,0.4387513,89.783290,0.0073822740,0.002911748,2.5353407,0.01123380,0.2142897
2,ENSG00000000457.14,0.0064341228,4.558421,2.860594624,0.09077394,0.5978506,37.830775,0.0072802746,0.003962031,1.8375108,0.06613452,1.0000000
3,ENSG00000000460.17,0.0050220029,4.899837,0.490188734,0.48384313,0.8873995,48.618616,0.0058351384,0.007092540,0.8227149,0.41067016,0.7828886
4,ENSG00000000971.15,-0.0041369648,4.485711,0.104956409,0.74596092,0.9588860,36.113081,-0.0033178092,0.013302673,-0.2494092,0.80304426,1.0000000
5,ENSG00000001036.13,0.0005079542,2.244848,0.004432601,0.94691784,0.9918812,6.524913,0.0008901366,0.008589560,0.1036300,0.91746294,1.0000000
6,ENSG00000001084.12,-0.0007446336,5.679795,0.035641324,0.85025811,0.9766942,84.992343,0.0001136087,0.003934678,0.0288737,0.97696532,0.9942858


In [12]:
print("GLM or DESeq2")
CompDF <- merge(fRes,data.frame(fDESeqRes),by = "row.names") %>% dplyr::select(Row.names, PValue, FDR, logFC, pvalue, padj, log2FoldChange)
CompDF$modp.GLM <- -log10(CompDF$PValue)*sign(CompDF$logFC)
CompDF$modp.DESeq <- -log10(CompDF$pvalue)*sign(CompDF$log2FoldChange)
if (nrow(CompDF[(is.na(CompDF$FDR)),]) > 0) {
    CompDF[(is.na(CompDF$FDR)),]$FDR <- 1
}
if (nrow(CompDF[(is.na(CompDF$padj)),]) > 0) {
    CompDF[(is.na(CompDF$padj)),]$padj <- 1
}
CompDF$ColorBlock <- rep("Niether", nrow(CompDF))
CompDF[CompDF$PValue < 0.05,]$ColorBlock <- "Adjusted"
CompDF[CompDF$pvalue < 0.05,]$ColorBlock <- "Not_Adjusted"
CompDF[(CompDF$pvalue < 0.05) & (CompDF$PValue < 0.05),]$ColorBlock <- "Both"
CompDF <- CompDF[CompDF$ColorBlock != "Niether",]
ourLine <- lm(CompDF$modp.DESeq~CompDF$modp.GLM-1)
summary(ourLine)$coefficients[1]
pdf(glue("/scratch/scjp_root/scjp1/kgmoo/analysis/20240814_kgmoo_RNAge/plots/CompareGLMvDESeqNomP_FAP_Fe.pdf"), width=6, height=6)
    mygg <- ggplot(data=CompDF, aes(x=-log10(PValue)*sign(logFC), y=-log10(pvalue)*sign(log2FoldChange))) +
        geom_point(alpha = 0.1)+#, position = position_jitter(width = 0.01)) +
        geom_abline(intercept = 0, slope = 1, alpha = 1, col = "black") +
        geom_abline(intercept = 0, slope = summary(ourLine)$coefficients[1], alpha = 1, col = "red")+
        theme_bw() + xlim(-20,20) + ylim(-20,20) + 
        xlab("Age Effects w/ GLMLRT -log10(Nominal P-value)*sign(log2FC)") + ylab("Age Effects w/ DESeq2 -log10(Nominal P-value)*sign(Coef)") + ggtitle(glue("Comparing Nominal P-values GLMLRT vs DESeq2")) 
   print(mygg)
dev.off()

[1] "GLM or DESeq2"


[1] 0.9821532

png 
  2

In [16]:
print("All or Just Old")
CompDF <- merge(data.frame(fDESeqRes), data.frame(fDESeqResOld),by = "row.names") %>% dplyr::select(Row.names, pvalue.x, padj.x, log2FoldChange.x, pvalue.y, padj.y, log2FoldChange.y)
CompDF$modp.Yng <- -log10(CompDF$pvalue.x)*sign(CompDF$log2FoldChange.x)
CompDF$modp.Old <- -log10(CompDF$pvalue.y)*sign(CompDF$log2FoldChange.y)
if (nrow(CompDF[(is.na(CompDF$padj.x)),]) > 0) {
    CompDF[(is.na(CompDF$padj.x)),]$padj.x <- 1
}
if (nrow(CompDF[(is.na(CompDF$padj.y)),]) > 0) {
    CompDF[(is.na(CompDF$padj.y)),]$padj.y <- 1
}
CompDF$ColorBlock <- rep("Niether", nrow(CompDF))
CompDF[CompDF$pvalue.x < 0.05,]$ColorBlock <- "Adjusted"
CompDF[CompDF$pvalue.y < 0.05,]$ColorBlock <- "Not_Adjusted"
CompDF[(CompDF$pvalue.y < 0.05) & (CompDF$pvalue.x < 0.05),]$ColorBlock <- "Both"
CompDF <- CompDF[CompDF$ColorBlock != "Niether",]
ourLine <- lm(CompDF$modp.y~CompDF$modp.x-1)
summary(ourLine)$coefficients[1]
pdf(glue("/scratch/scjp_root/scjp1/kgmoo/analysis/20240814_kgmoo_RNAge/plots/CompareYoungExclusionNomP_Type2a_Fe.pdf"), width=6, height=6)
    mygg <- ggplot(data=CompDF, aes(x=-log10(pvalue.x)*sign(log2FoldChange.x), y=-log10(pvalue.y)*sign(log2FoldChange.y))) +
        geom_point(alpha = 0.1)+#, position = position_jitter(width = 0.01)) +
        geom_abline(intercept = 0, slope = 1, alpha = 1, col = "black") +
        geom_abline(intercept = 0, slope = summary(ourLine)$coefficients[1], alpha = 1, col = "red")+
        theme_bw() + xlim(-20,20) + ylim(-20,20) + 
        xlab("Age Effects w/ All Samples -log10(Nominal P-value)*sign(log2FC)") + ylab("Age Effects w/ Younger than 40 removed -log10(Nominal P-value)*sign(Coef)") + ggtitle(glue("Comparing Nominal P-values w/wo young sample exclusion")) 
   print(mygg)
dev.off()

[1] "All or Just Old"


[1] 0.8429064

png 
  2